##### Copyright 2019 DeepMind Technologies Limited.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Frame Stacking with Reverb (numpy, in-process)

Frame stacking is a common technique in reinforcement learning where several
recent frames are combined into a single stacked observation. This notebook
shows how to stack frames **before** sending them to Reverb and how to store
sequences of stacked frames as sampleable items.

It uses Reverb's embedded / numpy-only mode (`Server(in_process=True)`); no
TensorFlow is required.

# Setup

In [ ]:
!pip install numpy dm-tree portpicker

In [ ]:
from collections import deque

import numpy as np
import reverb

A simple frame generator. Each frame is `[width, height]` uint8, filled with
its index so we can visually verify stacking order.

In [ ]:
FRAME_SHAPE = (16, 16)  # [width, height]
FRAME_DTYPE = np.uint8


def frame_generator(max_num_frames: int = 1000):
  for i in range(1, max_num_frames + 1):
    yield np.ones(FRAME_SHAPE, dtype=FRAME_DTYPE) * i

# Store stacked frames

`store_stacked` stacks the `stack_size` most recent frames, appends each stack
to a `TrajectoryWriter`, and once `sequence_length` stacks have been written it
creates a sampleable item referencing the last `sequence_length` stacks.

* If `stride` < `stack_size` then stacks overlap (some frames stored twice).
* If `stride` == `stack_size` then stacks are adjacent.
* If `stride` > `stack_size` then frames between stacks are dropped.

In [ ]:
def store_stacked(stack_size: int, stride: int, sequence_length: int):
  """Stacks frames before sending them to Reverb.

  Args:
    stack_size: The number of frames to stack.
    stride: The number of frames between each stack is created.
    sequence_length: The number of stacks in each sampleable item.
  """
  server = reverb.Server([reverb.Table.queue('stacked_frames', 100)],
                         in_process=True)
  client = server.in_process_client

  with client.trajectory_writer(table='stacked_frames',
                                num_keep_alive_refs=sequence_length) as writer:
    # Circular buffer of the `stack_size` most recent frames.
    buffer = deque(maxlen=stack_size)

    for i, frame in enumerate(frame_generator(5 * stride * sequence_length)):
      buffer.append(frame)

      # We can't insert anything before the first stack is full.
      if len(buffer) < stack_size or (i + 1) % stride != 0:
        continue

      # Stack the frames in buffer and append to the writer. The shape of
      # the stack is [stack_size, width, height].
      writer.append(np.stack(buffer))

      # If `sequence_length` full stacks have been written then insert an
      # item that can be sampled.
      stacks_written = (i + 1) // stride - (stack_size - 1) // stride
      if stacks_written >= sequence_length:
        writer.create_item(
            table='stacked_frames',
            trajectory=writer.history[-sequence_length:],
            priority=1.0)

    writer.flush()

  # Sample sequences of stacked frames. Each sample's data is a flat list of
  # columns; here there is one column whose shape is
  # [sequence_length, stack_size, width, height].
  for sequence in client.sample('stacked_frames', num_samples=2,
                                emit_timesteps=False):
    stacked = np.asarray(sequence.data[0])
    print('sampled shape:', stacked.shape)
    # The first stack of the first sampled sequence.
    print('first stack frame ids:', stacked[0, :, 0, 0].tolist())

  server.stop()

# Examples

## Adjacent stacks (no overlap)

4 frames per stack, stride 4, sequences of 3 stacks. The first 16 frames
yield two sampleable items:

```
[1 .. 16]
  -> [[1,2,3,4], [5,6,7,8], [9,10,11,12]]
  -> [[5,6,7,8], [9,10,11,12], [13,14,15,16]]
```

In [ ]:
store_stacked(stack_size=4, stride=4, sequence_length=3)

## Overlapping stacks (frames shared)

4 frames per stack, stride 2, sequences of 3 stacks. Since we stack **before**
sending to Reverb, most stacks are stored twice (double storage before any
compression). The first 12 frames yield three sampleable items:

```
[1 .. 12]
  -> [[1,2,3,4], [3,4,5,6], [5,6,7,8]]
  -> [[3,4,5,6], [5,6,7,8], [7,8,9,10]]
  -> [[5,6,7,8], [7,8,9,10], [9,10,11,12]]
```

In [ ]:
store_stacked(stack_size=4, stride=2, sequence_length=3)

## Dropped frames

2 frames per stack, stride 3, sequences of 3 stacks. Some frames are dropped
between stacks.

In [ ]:
store_stacked(stack_size=2, stride=3, sequence_length=3)